# Демонстрация метода

Ноутбук показывает суть расчета, который используется в приложении: загрузка зданий района, отбор жилой застройки, построение Weighted K-means и расчет покрытия по прямому расстоянию.

## 1. Импорт библиотек

In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import osmnx as ox
from sklearn.cluster import KMeans
from shapely.geometry import Point

## 2. Параметры эксперимента

In [ ]:
district_name = "Admiralteysky District, Saint Petersburg, Russia"
walking_time_min = 30
centers_count = 2

speed_kmh = 4.5
meters_per_min = speed_kmh * 1000 / 60
max_dist = walking_time_min * meters_per_min

## 3. Загрузка зданий из OpenStreetMap

In [ ]:
buildings = ox.features_from_place(district_name, tags={"building": True}).to_crs(epsg=3857)
buildings[["building", "geometry"]].head()

## 4. Отбор жилых зданий и расчет весов

In [ ]:
residential_types = {
    "residential", "apartments", "house", "detached", "semidetached_house",
    "terrace", "dormitory", "bungalow", "static_caravan"
}

residential = buildings[buildings["building"].astype(str).str.lower().isin(residential_types)].copy()
residential = residential[residential.geometry.notna() & residential.geometry.is_valid]

def parse_numeric(value):
    if pd.isna(value):
        return np.nan
    return pd.to_numeric(str(value).split(";")[0].replace("m", ""), errors="coerce")

levels = residential.get("building:levels", pd.Series(index=residential.index, dtype=float)).apply(parse_numeric)
levels = levels.fillna(1).clip(lower=1)
weights = residential.geometry.area.clip(lower=1) * levels
coords = np.array([[geom.centroid.x, geom.centroid.y] for geom in residential.geometry])

len(residential), coords.shape

## 5. Weighted K-means

In [ ]:
k = min(centers_count, len(coords))
kmeans = KMeans(n_clusters=k, n_init=10, random_state=42)
kmeans.fit(coords, sample_weight=weights.to_numpy())
centers = kmeans.cluster_centers_
centers

## 6. Приближенная оценка покрытия

In [ ]:
diffs = coords[:, None, :] - centers[None, :, :]
dists = np.sqrt((diffs ** 2).sum(axis=2))
coverage_pct = (dists.min(axis=1) <= max_dist).mean() * 100
coverage_pct

## 7. Визуализация результата

In [ ]:
centers_gdf = gpd.GeoDataFrame(
    {"center_id": range(1, len(centers) + 1)},
    geometry=[Point(x, y) for x, y in centers],
    crs="EPSG:3857",
).to_crs(epsg=4326)

centers_gdf